# 🎓 Placify — AI-Driven Placement Pipeline
### A Live Demonstration for the Project Guide

> **Placify** uses a **Hybrid 3-Stage Matching Engine** to intelligently map student profiles to the best-suited companies:

```
  INPUT: Resume PDF + Student Preferences
       ↓
   Stage 1 — Hard Filter  (Location / CTC)
       ↓
   Stage 2 — TF-IDF + Cosine Similarity
       ↓
   Stage 3 — Gemini LLM Semantic Re-ranking
       ↓
  OUTPUT: Top 5 Jobs + Readiness Score + PDF Report
```

---


## ⚙️ Cell 1 — Setup & Libraries
Import all dependencies and configure global display settings.

In [ ]:
import os, sys, json, ast, warnings, textwrap
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from IPython.display import display, IFrame
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from fpdf import FPDF
from google import genai

warnings.filterwarnings('ignore')

# Global chart style for professional presentation
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({
    'axes.titlesize'  : 14,
    'axes.titleweight': 'bold',
    'figure.dpi'      : 110,
})
PALETTE   = 'viridis'
FIG_W, FIG_H = 12, 5

print('✅ All libraries loaded successfully.')


## 🔑 Cell 2 — Load Paths & API Keys
Define folder paths and initialise the Gemini client from the `.env` file.

In [ ]:
from dotenv import load_dotenv

BASE_DIR    = Path('.').resolve().parent
COMPANY_DIR = BASE_DIR / 'company_dataset'
RESUME_DIR  = BASE_DIR / 'web_data' / 'resume'
OUTPUT_DIR  = BASE_DIR / 'web_data' / 'analysis'
REPORT_DIR  = BASE_DIR / 'web_data' / 'pdf'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load .env and initialise Gemini
env_path = BASE_DIR / 'venv' / '.env'
load_dotenv(dotenv_path=env_path)
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY', '')

gemini_client = None
if GEMINI_API_KEY:
    try:
        gemini_client = genai.Client(api_key=GEMINI_API_KEY)
        print(f'✅ Gemini client initialised.')
    except Exception as e:
        print(f'⚠️  Gemini init failed: {e}')
else:
    print('⚠️  GEMINI_API_KEY not found in .env — LLM stage will be skipped.')

print(f'   Base dir   : {BASE_DIR}')
print(f'   Companies  : {COMPANY_DIR}')
print(f'   Resumes    : {RESUME_DIR}')
print(f'   Output dir : {OUTPUT_DIR}')
print(f'   Report dir : {REPORT_DIR}')


## 📂 Cell 3 — Load Company Dataset
Read `companies.json` from the dataset folder and inspect its structure.

In [ ]:
companies_df = pd.DataFrame()

company_file = COMPANY_DIR / 'companies.json'
if not company_file.exists():
    print('❌ companies.json not found — check company_dataset/ folder.')
else:
    raw = json.loads(company_file.read_text(encoding='utf-8'))
    companies_df = pd.DataFrame(raw if isinstance(raw, list) else [raw])
    print(f'✅ Loaded {len(companies_df)} companies.')
    print(f'   Columns: {list(companies_df.columns)}')
    display(companies_df[['name', 'role', 'location', 'skills']].head(5))


## 📊 Cell 4 — Visualising the Central India Market
Three exploratory charts to understand the dataset before filtering:
- **Top 10 in-demand skills** across all companies.
- **Location distribution** of opportunities.
- **CTC distribution** histogram.

In [ ]:
# ── Chart 1: Top 10 Skills ────────────────────────────────────────
if 'skills' in companies_df.columns:
    all_skills = []
    for s in companies_df.get('skills', pd.Series([])).dropna():
        if isinstance(s, list):
            all_skills.extend([x.strip() for x in s])
        elif isinstance(s, str):
            try: all_skills.extend(ast.literal_eval(s))
            except: all_skills.extend([x.strip() for x in s.split(',')])

    skills_cnt = Counter(all_skills).most_common(10)
    sk, sv = zip(*skills_cnt) if skills_cnt else ([], [])

    fig1, ax1 = plt.subplots(figsize=(FIG_W, FIG_H))
    sns.barplot(ax=ax1, y=list(sk), x=list(sv), hue=list(sk), palette='viridis', legend=False)
    ax1.set_title('🔧 Top 10 Most In-Demand Tech Skills')
    ax1.set_xlabel('Number of Companies Requiring This Skills')
    ax1.set_ylabel('')
    for i, v in enumerate(sv): ax1.text(v + 0.1, i, str(v), va='center', fontsize=9)
    plt.tight_layout()
    display(fig1); plt.close(fig1)

In [ ]:
    # ── Chart 2: Top 10 Roles ────────────────────────────────────────
    if 'role' in companies_df.columns:
        role_cnt = companies_df['role'].dropna().value_counts().head(10)
        rk, rv = list(role_cnt.index), list(role_cnt.values)

        fig1, ax1 = plt.subplots(figsize=(FIG_W, FIG_H))
        sns.barplot(ax=ax1, y=rk, x=rv, hue=rk, palette='viridis', legend=False)
        ax1.set_title('🔧 Top 10 Most In-Demand Tech Roles')
        ax1.set_xlabel('Number of Companies')
        ax1.set_ylabel('')
        for i, v in enumerate(rv):
            ax1.text(v + 0.1, i, str(v), va='center', fontsize=9)
        plt.tight_layout()
        display(fig1); plt.close(fig1)


In [ ]:
    # ── Chart 3: Location Distribution ──────────────────────────────────
    if 'location' in companies_df.columns:
        lc = companies_df['location'].fillna('Unknown').value_counts().head(8)
        fig2, ax2 = plt.subplots(figsize=(FIG_W, FIG_H))
        sns.barplot(ax=ax2, y=lc.index, x=lc.values, hue=lc.index, palette='rocket', legend=False)
        ax2.set_title('📍 Job Locations — Top 8')
        ax2.set_xlabel('Number of Companies')
        ax2.set_ylabel('')
        for i, v in enumerate(lc.values): ax2.text(v + 0.1, i, str(v), va='center', fontsize=9)
        plt.tight_layout()
        display(fig2); plt.close(fig2)

## 🎒 Cell 5 — Load Student Data & Select Resume
Define the student's preferences (simulating what the quiz collects in production)  
and pick the resume to analyse by index.

In [ ]:
# ── List available resumes ──────────────────────────────────────────────
TARGET_RESUME_INDEX = 2   # ← change this to 1, 2, … to pick a different resume

In [ ]:
resume_files = sorted(RESUME_DIR.glob('*.pdf')) if RESUME_DIR.exists() else []
if not resume_files:
    print('❌ No resumes found in web_data/resume/')
else:
    print(f'📄 Available resumes in web_data/resume/:')
    for i, f in enumerate(resume_files):
        marker = '  ◀ SELECTED' if i == TARGET_RESUME_INDEX else ''
        print(f'   [{i}] {f.name}{marker}')

    selected_resume = resume_files[min(TARGET_RESUME_INDEX, len(resume_files) - 1)]

## 📖 Cell 6 — Resume Text Extraction
Use `pypdf` to extract raw text from the selected resume PDF.

In [ ]:
import re
resume_text = ''

def clean_pdf_text(raw: str) -> str:
    """Fix common pypdf artefacts: broken words across lines, excess spaces."""
    # Join lines broken mid-word (e.g. 'A\nSSISTANT' -> 'ASSISTANT')
    text = re.sub(r'(?<=[A-Za-z])\n(?=[A-Za-z])', '', raw)
    # Collapse runs of spaces
    text = re.sub(r' {2,}', ' ', text)
    # Remove blank lines
    text = '\n'.join(l for l in text.splitlines() if l.strip())
    return text.strip()

if 'selected_resume' not in dir() or not selected_resume:
    print('❌ No resume selected — run Cell 5 first.')
else:
    try:
        reader = PdfReader(str(selected_resume))
        raw_text   = '\n'.join(p.extract_text() or '' for p in reader.pages)
        resume_text = clean_pdf_text(raw_text)
        print(f'✅ Extracted {len(resume_text)} characters ({len(resume_text.split())} words) from:')
        print(f'   📄 {selected_resume.name}')
        print(f'\n─── First 600 characters preview ─────────────────────────')
        print(resume_text[:600])
        print('───────────────────────────────────────────────────────────')
    except Exception as e:
        print(f'❌ Failed to read PDF: {e}')


## 🔴 Cell 7 — Stage 1: Hard Filter (The Elimination Round)
Instantly eliminate companies that don't match the student's basic requirements  
— **Location** and **minimum CTC**. This reduces the search space dramatically.

In [ ]:
PREFERRED_LOCATION  = 'Indore','Remote'  # or 'Bhopal', 'Remote', or '' for no filter

In [ ]:
stage1_df = pd.DataFrame()

if companies_df.empty:
    print('❌ No company data — run Cell 3 first.')
else:
    before   = len(companies_df)
    filtered = companies_df.copy()

    # ── Normalise PREFERRED_LOCATION (supports string or tuple/list) ──
    _raw = PREFERRED_LOCATION if 'PREFERRED_LOCATION' in dir() else ''
    if isinstance(_raw, (list, tuple)):
        loc_list = [str(l).strip().lower() for l in _raw if str(l).strip()]
    else:
        loc_list = [str(_raw).strip().lower()] if str(_raw).strip() else []

    # ── Location filter (Remote always included) ──────────────────────
    if loc_list and 'location' in filtered.columns:
        loc_col = filtered['location'].fillna('').str.lower()
        mask = loc_col.str.contains('remote')
        for loc in loc_list:
            mask = mask | loc_col.str.contains(loc)
        filtered = filtered[mask]

    stage1_df = filtered.reset_index(drop=True)
    after     = len(stage1_df)
    label     = ', '.join(l.title() for l in loc_list) if loc_list else '(no filter — all companies)'

    print('🔴 Stage 1 — Location Filter Results')
    print(f'   Location preference : {label}')
    print(f'   Companies BEFORE    : {before}')
    print(f'   Companies AFTER     : {after}')
    print(f'   Eliminated          : {before - after} companies')

    if stage1_df.empty:
        print('\n⚠️ All companies filtered out — using full company set.')
        stage1_df = companies_df.copy()
    else:
        display(stage1_df[['name', 'role', 'location']].head(5))


## 🟡 Cell 8 — Stage 2: TF-IDF & Cosine Similarity
Vectorise every company's job description and tech stack alongside the resume text,  
then rank them mathematically by cosine similarity. Isolate the **Top 15** candidates.

In [ ]:
stage2_df = pd.DataFrame()

def build_company_text(row) -> str:
    parts = []
    for col in ('role', 'description'):
        val = str(row.get(col, ''))
        if val and val != 'nan': parts.append(val)
    skills = row.get('skills', [])
    if isinstance(skills, list): parts.extend(skills)
    elif isinstance(skills, str):
        try: parts.extend(ast.literal_eval(skills) if isinstance(ast.literal_eval(skills), list) else [skills])
        except: parts.append(skills)
    loc = str(row.get('location', ''))
    if loc and loc != 'nan': parts.append(loc)
    return ' '.join(parts).lower()

if stage1_df.empty or not resume_text:
    print('❌ Run Cells 5–7 before this cell.')
else:
    corpus = stage1_df.fillna('').apply(build_company_text, axis=1).tolist()
    all_texts = corpus + [resume_text.lower()]

    vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), max_features=8000)
    tfidf_mat  = vectorizer.fit_transform(all_texts)

    sims = cosine_similarity(tfidf_mat[-1], tfidf_mat[:-1]).flatten()

    stage2_df = stage1_df.copy()
    stage2_df['tfidf_score'] = sims
    stage2_df = stage2_df.sort_values('tfidf_score', ascending=False).head(15).reset_index(drop=True)

    print(f'🟡 Stage 2 — TF-IDF Ranking Complete')
    print(f'   Top 15 matches identified from {len(stage1_df)} filtered companies.')
    display(stage2_df[['name', 'role', 'location', 'tfidf_score']].head(5)
            .style.background_gradient(cmap='YlGn', subset=['tfidf_score']))


## 📊 Cell 9 — Visualising Stage 2 Insights
Two charts to understand TF-IDF results visually:
- **Top 15 companies by cosine similarity score.**
- **Most frequent words in Resume vs Top 15 Job Descriptions.**

In [ ]:
if stage2_df.empty:
    print('⚠️ Stage 2 data not available — run Cell 8 first.')
else:
    # ── Chart 1: Top 15 Cosine Similarity Bar ───────────────────────────
    top15 = stage2_df.copy()
    top15['label'] = top15['name'] + '  (' + top15['role'].str[:26] + ')'

    fig1, ax1 = plt.subplots(figsize=(FIG_W + 1, 7))
    bars = ax1.barh(top15['label'][::-1], top15['tfidf_score'][::-1],
                   color=sns.color_palette('YlOrRd', 15)[::-1], edgecolor='white', height=0.65)
    for bar, val in zip(bars, top15['tfidf_score'][::-1]):
        ax1.text(bar.get_width() + 0.0005, bar.get_y() + bar.get_height() / 2,
                 f'{val:.4f}', va='center', fontsize=9)
    ax1.set_xlabel('TF-IDF Cosine Similarity Score')
    ax1.set_title('🏆 Top 15 Companies — Resume Match Scores (Stage 2)', fontsize=14, fontweight='bold')
    ax1.set_xlim(0, top15['tfidf_score'].max() * 1.22)
    plt.tight_layout()
    display(fig1); plt.close(fig1)

    # ── Chart 2: Word Frequency Comparison ───────────────────────────────
    def top_words(text, n=12):
        vect = TfidfVectorizer(stop_words='english', max_features=100)
        vect.fit_transform([text])
        freq = Counter(text.lower().split())
        stop = vect.get_stop_words()
        cleaned = {w: c for w, c in freq.items() if w not in stop and w.isalpha() and len(w) > 3}
        return Counter(cleaned).most_common(n)

    jd_text  = ' '.join(stage2_df['description'].fillna('').tolist() + stage2_df['role'].fillna('').tolist())
    res_words = dict(top_words(resume_text))
    jd_words  = dict(top_words(jd_text))
    all_w = sorted(set(res_words) | set(jd_words), key=lambda w: -(res_words.get(w, 0) + jd_words.get(w, 0)))[:12]

    idx    = range(len(all_w))
    width  = 0.38
    res_v  = [res_words.get(w, 0) for w in all_w]
    jd_v   = [jd_words.get(w, 0)  for w in all_w]

    fig2, ax2 = plt.subplots(figsize=(FIG_W + 2, 5))
    ax2.bar([i - width/2 for i in idx], res_v, width, label='Resume',          color='#2980b9', edgecolor='white')
    ax2.bar([i + width/2 for i in idx], jd_v,  width, label='Top-15 JD Text', color='#e67e22', edgecolor='white')
    ax2.set_xticks(list(idx))
    ax2.set_xticklabels(all_w, rotation=30, ha='right', fontsize=9)
    ax2.set_ylabel('Raw Frequency')
    ax2.set_title('📝 Word Frequency Overlap: Resume vs Job Descriptions', fontsize=13, fontweight='bold')
    ax2.legend()
    plt.tight_layout()
    display(fig2); plt.close(fig2)


## 🟢 Cell 10 — Stage 3: LLM Semantic Re-ranking
Send the Top 15 TF-IDF matches and the full resume text to **Gemini 2.5 Flash**.  
The LLM semantically understands context and returns the final **Top 5** recommendations  
plus a Readiness Score, Strengths, Skill Gaps and Action Plan — all in strict JSON.

In [ ]:
import io, contextlib
from app.services.ai_service import analyze_profile

llm_report = None

if stage2_df.empty or not resume_text:
    print('❌ Run Cells 5–9 before this cell.')
else:
    candidates_json = json.dumps(
        stage2_df[['name', 'role', 'skills', 'description', 'location', 'email']]
        .fillna('').to_dict(orient='records')
    )
    loc_label = PREFERRED_LOCATION if isinstance(PREFERRED_LOCATION, str) else ', '.join(PREFERRED_LOCATION)
    user_context = (
        '--- QUICK DEMO MODE ---\n'
        'Preferred location: ' + loc_label + '\n'
        'Matching based on resume text only.\n'
        '--- RESUME CONTENT ---\n'
        + resume_text[:4000] + '...'
    )

    print('🤖 Contacting AI service (Gemini → Groq → Ollama)...')
    try:
        _buf = io.StringIO()
        with contextlib.redirect_stdout(_buf):
            llm_report = analyze_profile(
                user_context     = user_context,
                candidates_json  = candidates_json,
                mode             = 'Quick Demo',
                answers          = {},
                resume_extracted = resume_text[:4000]
            )

        if llm_report:
            print('✅ AI report received!')
            print('   Candidate      :', llm_report.get('candidate_name', '?'))
            print('   Readiness Score:', llm_report.get('readiness_score', '?'), '/ 100')
            print()
            recs = llm_report.get('job_recommendations', [])
            if recs:
                print('🏆 Top', len(recs), 'Job Recommendations:')
                print('─' * 65)
                rec_rows = []
                for j, rec in enumerate(recs, 1):
                    print(f'  {j}. {rec.get("company", "?")} — {rec.get("role", "?")}')
                    print(f'     📍 {rec.get("location", "?")}  |  {rec.get("match", "")[:60]}...')
                    print()
                    rec_rows.append({'#': j, 'Company': rec.get('company','?'), 'Role': rec.get('role','?'), 'Location': rec.get('location','?')})
                display(pd.DataFrame(rec_rows).set_index('#'))
        else:
            print('❌ AI service returned no data — check API keys in .env')
    except Exception as e:
        print('❌ AI call failed:', e)


## 📊 Cell 11 — Visualising LLM Insights
Two charts generated from the LLM's response:
- **Readiness Score Gauge** — how placement-ready is this student?
- **Skill Gaps & Action Plan** — structured display of personalised advice.

In [ ]:
if not llm_report:
    print('⚠️ No LLM report available — run Cell 10 first.')
else:
    score = llm_report.get('readiness_score', 0)

    # ── Chart 1: Readiness Gauge ─────────────────────────────────────
    colour = '#2ecc71' if score >= 70 else ('#f39c12' if score >= 40 else '#e74c3c')
    label  = 'Ready 🟢' if score >= 70 else ('Developing 🟡' if score >= 40 else 'Needs Work 🔴')
    fig1, ax1 = plt.subplots(figsize=(10, 2.8))
    ax1.barh([''], [score],           color=colour,    height=0.5, edgecolor='white')
    ax1.barh([''], [100 - score], left=[score], color='#ecf0f1', height=0.5, edgecolor='white')
    ax1.text(min(score - 2, 95), 0, f'{score}/100',
             va='center', ha='right', fontsize=17, fontweight='bold', color='white')
    ax1.text(102, 0, label, va='center', fontsize=11)
    ax1.set_xlim(0, 115)
    ax1.set_title(
        f'Placement Readiness Score — {llm_report.get("candidate_name", "Student")}',
        fontsize=13, fontweight='bold', pad=10
    )
    ax1.yaxis.set_visible(False)
    ax1.set_xlabel('Score out of 100', labelpad=8)
    plt.tight_layout(pad=1.5)
    display(fig1); plt.close(fig1)

    # ── Chart 2: Insights Cards ──────────────────────────────────────
    strengths = llm_report.get('strengths', [])
    gaps      = llm_report.get('gaps', [])
    acts      = llm_report.get('action_plan', [])
    max_items = max(len(strengths), len(gaps), len(acts), 1)

    ROW_H   = 1.5      # height per item row
    HEADER  = 1.2      # height for section header row
    fig_h   = HEADER + max_items * ROW_H + 0.5

    fig2, axes = plt.subplots(1, 3, figsize=(18, fig_h))
    configs = [
        ('Strengths',   strengths, '#eafaf1', '#1e8449'),
        ('Skill Gaps',  gaps,      '#fef9e7', '#9a7d0a'),
        ('Action Plan', acts,      '#eaf4fb', '#1a5276'),
    ]
    for ax, (cat, items, bg, hcol) in zip(axes, configs):
        ax.set_facecolor(bg)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, HEADER + max_items * ROW_H)
        ax.axis('off')
        top = HEADER + max_items * ROW_H
        # Section header
        ax.text(0.5, top - 0.5, cat, ha='center', va='center',
                fontsize=13, fontweight='bold', color='white',
                bbox=dict(boxstyle='round,pad=0.4', facecolor=hcol, edgecolor='none'))
        # Items — top to bottom
        for j, item in enumerate(items):
            y_pos = top - HEADER - j * ROW_H - ROW_H * 0.5
            wrapped = textwrap.fill(item, width=44)
            ax.text(0.06, y_pos, f'\u2022  {wrapped}',
                    va='center', fontsize=10, color='#1c2833',
                    linespacing=1.4, transform=ax.transData)

    plt.suptitle('Student Profile Insights — Generated by Gemini 2.5 Flash',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    display(fig2); plt.close(fig2)


## 📄 Cell 12 — Final PDF Report Generation
Generate a professional placement report PDF using `fpdf`  
and render it inline so the project guide can view it without leaving the browser.

In [ ]:
import time

pdf_path = None

if not llm_report:
    print('⚠️ No LLM report — run Cell 10 first.')
else:
    class PlacifyPDF(FPDF):
        def header(self):
            self.set_fill_color(27, 54, 93)
            self.rect(0, 0, 210, 22, 'F')
            self.set_font('Helvetica', 'B', 16)
            self.set_text_color(255, 255, 255)
            self.set_y(6)
            self.cell(0, 10, 'Placify - Placement Assessment Report', align='C', ln=True)
            self.ln(12)
            self.set_text_color(0, 0, 0)

        def footer(self):
            self.set_y(-13)
            self.set_font('Helvetica', 'I', 8)
            self.set_text_color(150, 150, 150)
            self.cell(0, 8, 'Generated by Placify AI  |  Page ' + str(self.page_no()), align='C')

        def section_title(self, title):
            self.set_fill_color(240, 244, 255)
            self.set_font('Helvetica', 'B', 12)
            self.set_text_color(27, 54, 93)
            self.cell(0, 9, title, ln=True, fill=True)
            self.set_text_color(0, 0, 0)
            self.ln(2)

        def bullet_list(self, items, label='  -  '):
            self.set_font('Helvetica', '', 10)
            for item in items:
                self.multi_cell(0, 6, label + str(item))
            self.ln(3)

    pdf = PlacifyPDF()
    pdf.add_page()

    score = llm_report.get('readiness_score', 0)
    name  = llm_report.get('candidate_name', 'Student')
    pdf.set_font('Helvetica', 'B', 13)
    pdf.cell(0, 8, 'Candidate: ' + name, ln=True)
    pdf.set_font('Helvetica', '', 11)
    pdf.cell(0, 7, 'Placement Readiness Score: ' + str(score) + '/100', ln=True)
    pdf.ln(5)

    pdf.section_title('Strengths')
    pdf.bullet_list(llm_report.get('strengths', []))

    pdf.section_title('Skill Gaps')
    pdf.bullet_list(llm_report.get('gaps', []))

    pdf.section_title('Action Plan')
    pdf.bullet_list(llm_report.get('action_plan', []))

    pdf.section_title('Top 5 Job Recommendations')
    for i_rec, rec in enumerate(llm_report.get('job_recommendations', []), 1):
        pdf.set_font('Helvetica', 'B', 10)
        pdf.cell(0, 6, str(i_rec) + '. ' + rec.get('company', '?') + ' - ' + rec.get('role', '?'), ln=True)
        pdf.set_font('Helvetica', '', 10)
        pdf.multi_cell(0, 5.5, '   Location : ' + rec.get('location', '?') + '\n   Why match : ' + rec.get('match', '?'))
        pdf.ln(3)

    filename = 'Placement_Report_' + str(int(time.time())) + '.pdf'
    pdf_path  = str(REPORT_DIR / filename)
    pdf.output(pdf_path)

    print('✅ PDF Report saved!')
    print('   File : ' + filename)
    print('   Path : ' + pdf_path)
    display(IFrame('web_data/reports/' + filename, width=900, height=640))


## ✉️ Cell 13 — Select & View Email Draft
Choose one of the 5 AI-drafted cold emails (index 0–4) to preview its full text.

In [ ]:
# ── ✏️  SET THIS — index of job to preview email draft for (0–4) ──────
EMAIL_DRAFT_INDEX = 1
# ──────────────────────────────────────────────────────────────────────

if not llm_report:
    print('⚠️ No LLM report available — run Cell 10 first.')
else:
    recs = llm_report.get('job_recommendations', [])
    if not recs:
        print('⚠️ No job recommendations found in the report.')
    else:
        idx  = min(EMAIL_DRAFT_INDEX, len(recs) - 1)
        rec  = recs[idx]
        draft = rec.get('email_draft', 'No email draft available.')

        print(f'✉️  Email Draft for Recommendation #{idx + 1}')
        print(f'   Company  : {rec.get("company", "?")}')
        print(f'   Role     : {rec.get("role", "?")}')
        print(f'   Location : {rec.get("location", "?")}')
        print('\n' + '─' * 65)
        print(draft)
        print('─' * 65)


---
## ✅ Demo Complete!

| Cell | Stage | What happened |
|------|-------|---------------|
| 1–2  | Setup | Libraries, paths, API keys |
| 3–4  | EDA   | 94 companies loaded and visualised |
| 5–6  | Input | Resume PDF parsed, student prefs defined |
| 7    | 🔴 Stage 1 | Hard Filter — Location + CTC elimination |
| 8–9  | 🟡 Stage 2 | TF-IDF Cosine Similarity — Top 15 ranked |
| 10–11| 🟢 Stage 3 | Gemini LLM Re-ranking — Top 5 + Insights |
| 12   | Output | Professional PDF report generated & rendered |

> To test a different resume: change `TARGET_RESUME_INDEX` in **Cell 5** and re-run from Cell 6 onward.
